<a href="https://colab.research.google.com/github/Disha-naveen25/EdVergencex/blob/main/ai-tutor_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Tutor Chatbot with Gradio

In [1]:
!mkdir -p /content/TutorBot
%cd /content/TutorBot

/content/TutorBot


In [2]:
!pip install -q gradio openai python-dotenv

In [3]:
from google.colab import userdata

NEXUSAI_API_KEY = userdata.get("NEXUSAI_API_KEY")

In [4]:
from google.colab import userdata

api_key = userdata.get("NEXUSAI_API_KEY")

with open(".env", "w") as f:
    f.write(f"NEXUSAI_API_KEY={api_key}\n")

print(".env created successfully.")

.env created successfully.


In [5]:
%%writefile .env.example

NEXUSAI_API_KEY=your_api_key_here

Writing .env.example


In [6]:
%%writefile requirements.txt
gradio
openai
python-dotenv

Writing requirements.txt


In [7]:
%%writefile app.py

import os
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI


# ============================================================
# 1. Load environment variables
# ============================================================

load_dotenv()

NEXUSAI_API_KEY = os.getenv("NEXUSAI_API_KEY")

NEXUSAI_BASE_URL = "https://nexusapi.navigatelabs.ai"
NEXUAI_MODEL_ID = "nova-micro"


# ============================================================
# 2. Check API key
# ============================================================

if not NEXUSAI_API_KEY:
    raise ValueError(
        "NEXUSAI_API_KEY is not set. "
        "Please add it to your .env file."
    )


# ============================================================
# 3. Create OpenAI client
# ============================================================

client = OpenAI(
    api_key=NEXUSAI_API_KEY,
    base_url=NEXUSAI_BASE_URL,
)


# ============================================================
# 4. TutorBot system prompt
# ============================================================

SYSTEM_MESSAGE = """
You are TutorBot, a friendly personal AI tutor.
Your job is to help students understand technical and academic topics.
Explain concepts in simple language.

When appropriate:

1. Give a simple definition.
2. Explain the concept step by step.
3. Give a real-world or programming example.
4. Mention important exam points.
5. End with a short 'Remember' point.

If the student asks for an exam answer, structure the answer
with headings and bullet points.

If the student asks something they don't understand,
simplify the explanation instead of using unnecessarily
complicated terminology.

Be encouraging, clear and concise.

Do not pretend to know information that you do not know.
"""


# ============================================================
# 5. Chat function
# ============================================================

def chat(message, history):
    """
    Handles the conversation between the student and TutorBot.
    """

    # Start conversation with the system message
    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        }
    ]

    # --------------------------------------------------------
    # Convert Gradio history into OpenAI message format
    # --------------------------------------------------------

    for item in history:

        # Modern Gradio message format
        if isinstance(item, dict):

            role = item.get("role")
            content = item.get("content")

            if role in ["user", "assistant"] and isinstance(content, str):
                messages.append(
                    {
                        "role": role,
                        "content": content
                    }
                )

    # --------------------------------------------------------
    # Add current user message
    # --------------------------------------------------------

    messages.append(
        {
            "role": "user",
            "content": message
        }
    )

    # --------------------------------------------------------
    # Send conversation to NexusAI
    # --------------------------------------------------------

    response = client.chat.completions.create(
        model=NEXUAI_MODEL_ID,
        messages=messages
    )

    # --------------------------------------------------------
    # Extract assistant response
    # --------------------------------------------------------

    assistant_response = response.choices[0].message.content

    return assistant_response


# ============================================================
# 6. Clear chat function
# ============================================================

def clear_chat():
    return []


# ============================================================
# 7. Gradio Interface
# ============================================================

with gr.Blocks(
    title="TutorBot"
) as demo:

    gr.Markdown(
        """
        # 🎓 TutorBot
        ### Your Personal AI Study Tutor

        Ask questions about your subjects and get simple,
        clear and exam-oriented explanations.
        """
    )

    chatbot = gr.Chatbot(
        label="TutorBot",
        type="messages",
        height=500,
        value=[
            {
                "role": "assistant",
                "content": (
                    "Hi! 👋 I'm TutorBot.\n\n"
                    "Ask me anything you're studying. I can explain "
                    "concepts simply, give examples, and help you "
                    "prepare exam answers.\n\n"
                    "Try asking:\n"
                    "• What is Machine Learning?\n"
                    "• Explain inheritance\n"
                    "• What is an AI agent?\n"
                    "• Explain overfitting"
                )
            }
        ]
    )

    message_box = gr.Textbox(
        label="Ask TutorBot",
        placeholder="Type your question here...",
        lines=2
    )

    with gr.Row():

        send_button = gr.Button(
            "Send",
            variant="primary"
        )

        clear_button = gr.Button(
            "Clear Chat"
        )

    gr.Examples(
        examples=[
            "What is Machine Learning?",
            "Explain polymorphism in simple words",
            "What is overfitting?",
            "Explain TCP and UDP",
            "What is an AI agent?"
        ],
        inputs=message_box
    )

    # Send using button
    send_button.click(
        fn=chat,
        inputs=[message_box, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=message_box
    )

    # Send using Enter
    message_box.submit(
        fn=chat,
        inputs=[message_box, chatbot],
        outputs=chatbot
    ).then(
        fn=lambda: "",
        inputs=None,
        outputs=message_box
    )

    # Clear conversation
    clear_button.click(
        fn=clear_chat,
        inputs=None,
        outputs=chatbot
    )


# ============================================================
# 8. Launch application
# ============================================================

if __name__ == "__main__":
    demo.launch(
        share=True
    )

Writing app.py


In [8]:
%%writefile README.md

# 🎓 TutorBot

## Your Personal AI Study Tutor

TutorBot is a beginner-friendly AI chatbot built using Python and Gradio.

It acts as a personal study tutor and provides simple, clear and exam-oriented explanations for technical and academic topics.

---

## Features

- Ask academic and technical questions
- Simple explanations
- Step-by-step explanations
- Programming and real-world examples
- Exam-oriented points
- Conversation memory
- Follow-up questions
- Gradio chatbot interface
- NexusAI `nova-micro` model
- Secure API key configuration

---

## Technologies Used

- Python
- Gradio
- OpenAI Python Client
- NexusAI API
- python-dotenv

---

## Project Structure

```text
TutorBot/
│
├── app.py
├── requirements.txt
├── .env
├── .env.example
└── README.md

Writing README.md


In [9]:
pip install -r requirements.txt

In [11]:
%%writefile .env.example
NEXUSAI_API_KEY=your_api_key_here

Overwriting .env.example


In [12]:
from google.colab import userdata

api_key = userdata.get("NEXUSAI_API_KEY")

with open(".env", "w") as f:
    f.write(f"NEXUSAI_API_KEY={api_key}\n")

print("API key configured successfully.")

API key configured successfully.


In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API key loaded:", bool(os.getenv("NEXUSAI_API_KEY")))

API key loaded: True


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

print("API key loaded:", bool(os.getenv("NEXUSAI_API_KEY")))

API key loaded: True


In [15]:
from openai import OpenAI
import os

NEXUSAI_API_KEY = os.getenv("NEXUSAI_API_KEY")
NEXUSAI_BASE_URL = "https://nexusapi.navigatelabs.ai"
NEXUAI_MODEL_ID = "nova-micro"

client = OpenAI(
    api_key=NEXUSAI_API_KEY,
    base_url=NEXUSAI_BASE_URL,
)

print("NexusAI client created successfully.")
print("Model:", NEXUAI_MODEL_ID)

NexusAI client created successfully.
Model: nova-micro


In [16]:
response = client.chat.completions.create(
    model=NEXUAI_MODEL_ID,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ]
)

print(response.choices[0].message.content)

Hello!


In [17]:
SYSTEM_MESSAGE = "You are TutorBot"

messages = [
    {
        "role": "system",
        "content": SYSTEM_MESSAGE
    }
]

print("TutorBot conversation state created.")

TutorBot conversation state created.


In [18]:
def tutorbot_chat(user_message):
    # Add the student's message
    messages.append({
        "role": "user",
        "content": user_message
    })

    # Ask NexusAI
    response = client.chat.completions.create(
        model=NEXUAI_MODEL_ID,
        messages=messages
    )

    # Get TutorBot's reply
    assistant_message = response.choices[0].message.content

    # Save TutorBot's reply for conversation memory
    messages.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message

In [19]:
print(tutorbot_chat("What is ML?"))

Machine Learning (ML) is a subset of artificial intelligence (AI) that focuses on building systems that learn from data and improve their performance over time without being explicitly programmed. Here's a more detailed breakdown:

### Key Concepts of Machine Learning

1. **Data**: ML algorithms operate on data, which can be structured (like a database) or unstructured (like text or images).

2. **Models**: These are the mathematical representations created by ML algorithms. They can be linear models, decision trees, neural networks, etc.

3. **Training**: The process by which a model learns from data. The model is fed data and adjusts its parameters to better predict outcomes or classify data.

4. **Testing**: After training, the model is tested on a separate dataset to evaluate its performance. This helps in understanding how well the model generalizes to new, unseen data.

5. **Evaluation**: Various metrics like accuracy, precision, recall, F1 score, etc., are used to measure how we

In [24]:
import gradio as gr

def chat_with_tutor(message, history):
    reply = tutorbot_chat(message)

    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": reply
    })

    return history, ""


with gr.Blocks(title="TutorBot") as demo:

    gr.Markdown("""
    # 🎓 TutorBot
    ### Your Friendly AI Study Assistant

    Ask me anything you're studying. I can explain concepts simply,
    give examples, and help you prepare exam answers.
    """)

    chatbot = gr.Chatbot(
        label="TutorBot",
        height=500
    )

    msg = gr.Textbox(
        label="Ask TutorBot",
        placeholder="Ask a technical or academic question..."
    )

    send = gr.Button("Send", variant="primary")
    clear = gr.Button("Clear Chat")

    send.click(
        chat_with_tutor,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    msg.submit(
        chat_with_tutor,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    clear.click(
    clear_tutorbot,
    outputs=[chatbot]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8724e3661873554411.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
def clear_tutorbot():
    global messages

    messages = [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE
        }
    ]

    return []